In [1]:
!pip install -q ultralytics
!pip install -q pycocotools
!pip install -q torchmetrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 20.4 MB/s eta 0:00:00


In [2]:
import os
import cv2
import time
import yaml
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO

print("Torch Version :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Torch Version : 2.11.0+cu128
CUDA Available : True


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device : cuda
Tesla T4


In [4]:
!mkdir -p ~/.kaggle

from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"hajerahmmed","key":"9a48d1f9448005f959c30fa332472b8c"}'}

In [5]:
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!kaggle datasets download -d thedatasith/sku110k-annotations

Dataset URL: https://www.kaggle.com/datasets/thedatasith/sku110k-annotations
License(s): Attribution-NonCommercial-ShareAlike 3.0 IGO (CC BY-NC-SA 3.0 IGO)
100% 13.2G/13.2G [02:17<00:00, 103MB/s]



In [7]:
!unzip -q sku110k-annotations.zip

In [8]:
!unzip best.pt.zip

unzip:  cannot find or open best.pt.zip, best.pt.zip.zip or best.pt.zip.ZIP.


In [9]:
YOLO_MODEL = "/content/best.pt"

FASTER_MODEL = "/content/FasterRCNN_Final-2.pth"

DATASET = "/content/SKU110K_fixed"

In [10]:
import os

print(os.path.exists(YOLO_MODEL))
print(os.path.exists(FASTER_MODEL))
print(os.path.exists(DATASET))

True
True
True


In [11]:
from ultralytics import YOLO

yolo_model = YOLO(YOLO_MODEL)

print("YOLO Loaded Successfully ✅")

YOLO Loaded Successfully ✅


In [12]:
import torch
import torchvision

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = fasterrcnn_resnet50_fpn(weights=None)

num_classes = 2

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes
)

model.load_state_dict(
    torch.load(
        FASTER_MODEL,
        map_location=device
    )
)

model.to(device)

model.eval()

print("Faster R-CNN Loaded Successfully ✅")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 119MB/s]


Faster R-CNN Loaded Successfully ✅


In [17]:
yaml_content = """path: /content/SKU110K_fixed
train: images/train
val: images/val
test: images/test

nc: 1
names: ['object']
"""

with open("/content/data_kaggle.yaml", "w") as f:
    f.write(yaml_content)

print("تم التعديل ✅")

تم التعديل ✅


In [18]:
yolo_results = yolo_model.val(
    data="/content/data_kaggle.yaml",
    split="val",
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    verbose=True
)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3435.6±1343.8 MB/s, size: 887.1 KB)
val: Scanning /content/SKU110K_fixed/labels/val... 584 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 584/584 913.9it/s 0.6s
val: New cache created: /content/SKU110K_fixed/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 73/73 2.0it/s 36.5s
                   all        584      90456      0.856      0.772      0.825      0.479
Speed: 1.7ms preprocess, 5.4ms inference, 0.0ms loss, 5.1ms postprocess per image
Results saved to /content/runs/detect/val-2
